# Role-Based Collaboration (CrewAI-style) | Multi-Agent Collaboration

In [1]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from typing_extensions import NotRequired
from langchain_core.globals import set_llm_cache
from langchain_community.cache import SQLiteCache
from helper import plot_mermaid, stream_invoke

In [2]:
# Setup Response caching
set_llm_cache(SQLiteCache(database_path=".langchain_cache.db"))

In [3]:
model = ChatOpenAI(model="gpt-4o")

In [4]:
# Role definitions (CrewAI-style)
ROLES = {
    "researcher": {
        "role": "Senior Research Analyst",
        "goal": "Gather comprehensive, accurate information with citations",
        "backstory": "You are a 15-year veteran research analyst who has worked at top consulting firms. "
                     "You are meticulous about data accuracy and always provide sources for your claims.",
        "tool": "search_web",  # Unique capability: can search the web for information
    },
    "writer": {
        "role": "Content Strategist",
        "goal": "Transform research into engaging, well-structured content",
        "backstory": "You are an award-winning content strategist with a background in journalism. "
                     "You excel at making complex topics accessible while maintaining depth and accuracy.",
        "constraint": "Keep the article under 500 words. Brevity is your hallmark.",
    },
    "editor": {
        "role": "Senior Editor",
        "goal": "Polish content to publication quality with perfect grammar and flow",
        "backstory": "You are a senior editor with 20 years at top publications. You have an eagle eye "
                     "for grammar, consistency, and narrative flow. You also fact-check key claims.",
        "tool": "check_facts",  # Unique capability: verify claims against the research
    },
}

class CrewState(TypedDict):
    topic: str
    research: NotRequired[str]
    draft: NotRequired[str]
    final_article: NotRequired[str]

In [5]:
def search_web(query: str) -> str:
    """Researcher's unique tool — only the researcher role can search the web."""
    # In production: use DuckDuckGo, Tavily, or Google Search API
    return f"[Web results for '{query}']: Recent studies show AI agents are transforming workflows..."

def check_facts(claims: str, research: str) -> str:
    """Editor's unique tool — cross-reference claims against source research."""
    response = model.invoke(
        f"Fact-check these claims against the original research. Flag any unsupported claims.\n\n"
        f"Claims:\n{claims}\n\nResearch:\n{research}"
    )
    return response.content

def researcher_task(state: CrewState) -> dict:
    role = ROLES["researcher"]
    # Researcher uses its unique search_web tool to gather information
    search_results = search_web(state["topic"])
    response = model.invoke(
        f"You are: {role['role']}\n"
        f"Goal: {role['goal']}\n"
        f"Backstory: {role['backstory']}\n"
        f"Your unique tool: {role['tool']} (you have web search access)\n\n"
        f"Task: Research the following topic thoroughly. Provide key facts, statistics, "
        f"expert opinions, and current trends. Organize findings into clear sections.\n\n"
        f"Topic: {state['topic']}\n\n"
        f"Web search results:\n{search_results}"
    )
    return {"research": response.content}

def writer_task(state: CrewState) -> dict:
    role = ROLES["writer"]
    # Writer is constrained by its role's word limit — no tools, just disciplined writing
    response = model.invoke(
        f"You are: {role['role']}\n"
        f"Goal: {role['goal']}\n"
        f"Backstory: {role['backstory']}\n"
        f"CONSTRAINT: {role['constraint']}\n\n"
        f"Task: Using the research below, write an engaging article. "
        f"Include an attention-grabbing intro, well-structured body, and compelling conclusion.\n\n"
        f"Research:\n{state['research']}"
    )
    return {"draft": response.content}

def editor_task(state: CrewState) -> dict:
    role = ROLES["editor"]
    # Editor uses its unique check_facts tool to verify claims before finalizing
    fact_check_result = check_facts(state["draft"], state["research"])
    response = model.invoke(
        f"You are: {role['role']}\n"
        f"Goal: {role['goal']}\n"
        f"Backstory: {role['backstory']}\n"
        f"Your unique tool: {role['tool']} (you verified facts below)\n\n"
        f"Task: Edit and polish this article to publication quality. Fix grammar, improve flow, "
        f"and address any issues found during fact-checking.\n\n"
        f"Draft:\n{state['draft']}\n\n"
        f"Fact-check results:\n{fact_check_result}"
    )
    return {"final_article": response.content}

In [6]:
graph = StateGraph(CrewState)
graph.add_sequence([("researcher", researcher_task), ("writer", writer_task), ("editor", editor_task)])
graph.add_edge(START, "researcher")
graph.add_edge("editor", END)

crew = graph.compile()

In [7]:
# Plot the workflow
plot_mermaid(crew)

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	researcher(researcher)
	writer(writer)
	editor(editor)
	__end__([<p>__end__</p>]):::last
	__start__ --> researcher;
	researcher --> writer;
	writer --> editor;
	editor --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

In [8]:
result = crew.invoke({"topic": "The impact of AI agents on software development workflows in 2025"})
print(result["final_article"])

**The AI Revolution in Software Development: What to Expect by 2025**

Imagine a future where the repetitive and laborious aspects of software development are seamlessly managed by artificial intelligence (AI) tools, liberating developers to focus on innovation and creativity. This future is fast approaching. By 2025, the integration of AI agents into software development workflows is anticipated to transform the industry, according to recent studies and expert insights.

**Revamping Routine Tasks**

AI is reshaping the software development landscape by automating repetitive tasks that have historically consumed much of a developer's time. A study by Forrester indicates that by 2025, AI could manage up to 30% of standard coding tasks, dramatically reducing development cycles and minimizing human error. This shift not only boosts productivity but also empowers developers to engage in more complex and creative challenges.

**Elevating Code Quality and Prototyping**

With a keen eye for d

In [9]:
stream_invoke(crew, {"topic": "The impact of AI agents on software development workflows in 2025"})


────────────────────────────────────────────────────────────────────────────────
  STREAMING EXECUTION
────────────────────────────────────────────────────────────────────────────────

────────────────────────────────────────────────────────────────────────────────
  EXECUTION COMPLETE
────────────────────────────────────────────────────────────────────────────────



{'topic': 'The impact of AI agents on software development workflows in 2025',
 'research': '**The Impact of AI Agents on Software Development Workflows in 2025**\n\nIn recent years, the integration of Artificial Intelligence (AI) agents into software development workflows has undergone significant transformation. By 2025, AI is poised to further revolutionize software development processes, making them more efficient, productive, and innovative. This report explores key aspects of this impact, drawing on recent studies, expert opinions, and industry trends.\n\n**1. Key Improvements by AI Agents in Software Development**\n\n- **Automation of Routine Tasks**: AI agents are increasingly automating repetitive coding and testing tasks, allowing developers to focus on more complex issues. According to a study by Forrester, developers can expect AI to handle up to 30% of standard coding tasks, minimizing human error and speeding up the development cycle [Forrester Research, 2023].\n  \n- **E